# Neural Voice Identity Control & Deepfake Audio Analysis
**Disertație 2026 — Demo interactiv**

---

**⚠️ DISCLAIMER:**
Acest sistem este construit exclusiv în scop academic, pentru demonstrarea capacităților și limitelor
tehnologiilor de sinteză vocală și detecție deepfake audio. Vocile simulate sunt ale unor persoane publice
ale căror înregistrări sunt disponibile public. Sistemul **nu este** destinat înșelăciunii, uzului comercial
sau distribuției publice.

---

**Cum rulezi:**
1. Asigură-te că runtime-ul este T4 GPU (Runtime → Change runtime type)
2. Rulează toate celulele: Runtime → Run all
3. Accesează URL-ul Gradio generat la final

**Prerequisite:** Modelele fine-tuned (`.pth`) trebuie să existe în Google Drive.
Dacă nu ai antrenat încă, rulează `02_rvc_finetune.ipynb` prima dată.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU)"}')

In [ ]:
# Instalare dependinte
import subprocess, sys

def pip_install(pkg, extra_args=[]):
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg] + extra_args,
        capture_output=True, text=True
    )
    status = 'OK' if r.returncode == 0 else 'FAIL'
    print(f'  [{status}] {pkg}')
    return r.returncode == 0

print('Instalare pachete...')

for pkg in [
    'gradio>=4.7.0',
    'librosa==0.9.2',
    'soundfile',
    'pydub',
    'transformers',
    'accelerate',
    'matplotlib',
    'plotly',
    'ffmpeg-python',
    'av',
    'scipy',
    'numba',
    'einops',
    'praat-parselmouth>=0.4.2',
    'torchcrepe',
    'faiss-cpu',
    'local_attention',
    'demucs',
]:
    pip_install(pkg)

if not pip_install('pyworld'):
    pip_install('pyworld', ['--no-build-isolation'])

subprocess.run(['apt-get', 'install', '-q', '-y', 'ffmpeg'], capture_output=True)
print('Dependinte instalate.')

In [ ]:
# Setup RVC (clone if not present)
import os
import sys

RVC_DIR = '/content/RVC'

if not os.path.exists(RVC_DIR):
    print('Cloning RVC repository...')
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC_DIR} -q
    print('RVC cloned.')
else:
    print('RVC already present.')

sys.path.insert(0, RVC_DIR)

# Download HuBERT + RMVPE if needed
assets = [
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt',
     f'{RVC_DIR}/assets/hubert/hubert_base.pt'),
    ('https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt',
     f'{RVC_DIR}/assets/rmvpe/rmvpe.pt'),
]
for url, dest in assets:
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f'Downloading {os.path.basename(dest)}...')
        !wget -q '{url}' -O '{dest}'
    else:
        print(f'  OK: {os.path.basename(dest)}')
print('RVC setup complete.')

In [ ]:
# Mock fairseq cu transformers HuBERT
# RVC importa fairseq pentru a incarca HuBERT la inferenta.
# fairseq nu merge pe Python 3.12, deci cream un modul fals
# cu aceeasi interfata API dar care foloseste transformers intern.
import sys, types, torch

def _install_fairseq_mock():
    from transformers import HubertModel, Wav2Vec2FeatureExtractor

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    is_half = torch.cuda.is_available()

    print('Incarcare HuBERT pentru inferenta (transformers)...')
    _extractor = Wav2Vec2FeatureExtractor.from_pretrained('facebook/hubert-base-ls960')
    _hubert = HubertModel.from_pretrained('facebook/hubert-base-ls960').to(device)
    if is_half:
        _hubert = _hubert.half()
    _hubert.eval()
    print('HuBERT incarcat.')

    class FakeHubertModel:
        """Interfata fairseq, backend transformers. Layer 9 = acelasi ca la antrenare."""
        def extract_features(self, source, padding_mask=None, output_layer=9):
            with torch.no_grad():
                src = source.half() if is_half else source.float()
                out = _hubert(src, output_hidden_states=True)
                feats = out.hidden_states[9].float()
            return feats, None

        def final_proj(self, x): return x
        def to(self, d): return self
        def half(self): return self
        def float(self): return self
        def eval(self): return self

    _fake_model = FakeHubertModel()

    def _load_fake(*args, **kwargs):
        return [_fake_model], None, None

    # Inregistreaza modulele fake in sys.modules
    for mod_name in ['fairseq', 'fairseq.checkpoint_utils',
                     'fairseq.data', 'fairseq.models',
                     'fairseq.tasks', 'fairseq.criterions']:
        sys.modules[mod_name] = types.ModuleType(mod_name)

    sys.modules['fairseq.checkpoint_utils'].load_model_ensemble_and_task = _load_fake
    sys.modules['fairseq'].checkpoint_utils = sys.modules['fairseq.checkpoint_utils']
    print('Mock fairseq OK.')

_install_fairseq_mock()

## Configurare voci

Verifica ca modelele `.pth` exista pe Drive inainte sa continui.

In [ ]:
from pathlib import Path

# ============================================================
# CONFIGURARE
# ============================================================
DRIVE_BASE = '/content/drive/MyDrive/disertatie'
MODELS_DIR = Path(f'{DRIVE_BASE}/models')

VOICE_CONFIG = {
    'voice_1': {
        'name': 'Călin Georgescu',
        'model_file': 'voice_1.pth',
        'index_file': 'voice_1.index',
        'transpose': 0,
    },
    'voice_2': {
        'name': 'Nicușor Dan',
        'model_file': 'voice_2.pth',
        'index_file': 'voice_2.index',
        'transpose': 0,
    },
    'voice_3': {
        'name': 'Diana Șoșoacă',
        'model_file': 'voice_3.pth',
        'index_file': 'voice_3.index',
        'transpose': 0,
    },
}

# Ensemble de 4 modele deepfake cu arhitecturi diferite
DEEPFAKE_MODEL_ID   = 'MelodyMachine/Deepfake-audio-detection-V2'
DEEPFAKE_MODEL_ID_2 = 'garystafford/wav2vec2-deepfake-voice-detector'
DEEPFAKE_MODEL_ID_3 = 'DavidCombei/wavLM-base-Deepfake_V2'
DEEPFAKE_MODEL_ID_4 = 'Gustking/wav2vec2-large-xlsr-deepfake-audio-classification'
# ============================================================

available_voices = {}
for vid, cfg in VOICE_CONFIG.items():
    model_path = MODELS_DIR / cfg['model_file']
    status = '✓' if model_path.exists() else '✗ (lipsește — rulează 02_rvc_finetune.ipynb)'
    print(f'{status} {cfg["name"]} ({vid})')
    if model_path.exists():
        available_voices[vid] = cfg['name']

if not available_voices:
    print('\n⚠ Niciun model găsit. App-ul pornește dar VC nu va funcționa.')
else:
    print(f'\nVoci disponibile: {list(available_voices.values())}')

In [ ]:
# Export modele din format checkpoint (antrenare) → format inferenta (RVC)
# Ruleaza o singura data; daca modelele sunt deja exportate, nu face nimic.
import torch, shutil
from pathlib import Path

def _export_model(vid, cfg_obj, sr=40000):
    src = str(MODELS_DIR / cfg_obj['model_file'])
    tmp = f'/content/exp_{vid}.pth'
    if not Path(src).exists():
        print(vid + ': lipseste modelul, sari')
        return
    ckpt = torch.load(src, map_location='cpu')
    keys = list(ckpt.keys()) if isinstance(ckpt, dict) else []
    if 'weight' in keys and 'config' in keys:
        print(vid + ': deja in format corect')
        return
    if 'model' in keys:
        sd = ckpt['model']
    else:
        sd = ckpt
    weight = {}
    for k, v in sd.items():
        if 'enc_q' not in k:
            weight[k] = v.half()
    n_spk = weight['emb_g.weight'].shape[0]
    cfg_arr = [
        1025, 32,
        192, 192, 768, 2, 6, 3, 0,
        '1',
        [3, 7, 11],
        [[1, 3, 5], [1, 3, 5], [1, 3, 5]],
        [10, 10, 2, 2],
        512,
        [16, 16, 4, 4],
        n_spk,
        256,
        sr,
    ]
    opt = {
        'weight': weight,
        'config': cfg_arr,
        'info': 'exported',
        'sr': str(sr),
        'f0': 1,
        'version': 'v2',
    }
    torch.save(opt, tmp)
    shutil.copy2(tmp, src)
    check = torch.load(src, map_location='cpu')
    print(vid + ': exportat OK — chei: ' + str(list(check.keys())))

print('Exportare modele...')
for vid, cfg in VOICE_CONFIG.items():
    _export_model(vid, cfg)
print('Export complet.')

In [ ]:
import torch
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

def _load_df_model(model_id):
    extractor = AutoFeatureExtractor.from_pretrained(model_id)
    model = AutoModelForAudioClassification.from_pretrained(model_id)
    model.eval()
    fake_idx = next(
        (int(i) for i, lbl in model.config.id2label.items()
         if any(k in lbl.lower() for k in ('fake', 'spoof', 'ai', 'synth', 'generated'))),
        1
    )
    print(f'  Labels: {model.config.id2label} — fake_idx={fake_idx}')
    return extractor, model, fake_idx

print(f'[1/4] {DEEPFAKE_MODEL_ID}')
_df_extractor1, _df_model1, _fake_idx1 = _load_df_model(DEEPFAKE_MODEL_ID)

print(f'[2/4] {DEEPFAKE_MODEL_ID_2}')
_df_extractor2, _df_model2, _fake_idx2 = _load_df_model(DEEPFAKE_MODEL_ID_2)

print(f'[3/4] {DEEPFAKE_MODEL_ID_3}')
_df_extractor3, _df_model3, _fake_idx3 = _load_df_model(DEEPFAKE_MODEL_ID_3)

print(f'[4/4] {DEEPFAKE_MODEL_ID_4}')
_df_extractor4, _df_model4, _fake_idx4 = _load_df_model(DEEPFAKE_MODEL_ID_4)

print('Toate 4 modele deepfake incarcate.')

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import subprocess, tempfile
from pathlib import Path

SAMPLE_RATE_DF = 16000
WINDOW_S = 2.0
HOP_S = 0.5

# Culori per model in grafice
_MODEL_COLORS = ['#e74c3c', '#8e44ad', '#2980b9', '#27ae60']
_MODEL_LABELS = [
    'M1 MelodyMachine',
    'M2 Wav2Vec2-XLSR',
    'M3 WavLM-base',
    'M4 XLS-R-300M',
]

def _separate_vocals(audio_path):
    out_dir = tempfile.mkdtemp()
    result = subprocess.run(
        ['python', '-m', 'demucs', '--two-stems', 'vocals', '-o', out_dir, str(audio_path)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError('Demucs error: ' + result.stderr[-500:])
    for vocals_path in Path(out_dir).rglob('vocals.wav'):
        print('Voce separata: ' + str(vocals_path))
        return str(vocals_path)
    raise FileNotFoundError('Demucs: vocals.wav negasit in ' + out_dir)

def _predict_window(window, extractor, model, fake_idx):
    inputs = extractor(window, sampling_rate=SAMPLE_RATE_DF, return_tensors='pt', padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).squeeze().numpy()
    return float(probs[fake_idx])

def detect_deepfake(audio_path, separate_vocals=False):
    if separate_vocals:
        print('Separare vocala cu Demucs...')
        audio_path = _separate_vocals(audio_path)
        print('Analiza pe voce izolata.')

    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE_DF, mono=True)
    win = int(WINDOW_S * SAMPLE_RATE_DF)
    hop = int(HOP_S * SAMPLE_RATE_DF)

    timestamps = []
    scores_per_model = [[], [], [], []]
    models_params = [
        (_df_extractor1, _df_model1, _fake_idx1),
        (_df_extractor2, _df_model2, _fake_idx2),
        (_df_extractor3, _df_model3, _fake_idx3),
        (_df_extractor4, _df_model4, _fake_idx4),
    ]

    for start in range(0, max(1, len(audio) - win + 1), hop):
        w = audio[start:start + win]
        if len(w) < win:
            w = np.pad(w, (0, win - len(w)))
        for i, (ext, mdl, idx) in enumerate(models_params):
            scores_per_model[i].append(_predict_window(w, ext, mdl, idx))
        timestamps.append((start + win // 2) / SAMPLE_RATE_DF)

    if not timestamps:
        padded = np.pad(audio, (0, win - len(audio)))
        for i, (ext, mdl, idx) in enumerate(models_params):
            scores_per_model[i] = [_predict_window(padded, ext, mdl, idx)]
        timestamps = [len(audio) / (2 * SAMPLE_RATE_DF)]

    ts = np.array(timestamps)
    fs = [np.array(s) for s in scores_per_model]
    fs_ens = np.mean(fs, axis=0)

    weights = np.hanning(len(ts)) + 0.1
    global_scores = [float(np.average(f, weights=weights)) for f in fs]
    score_ens = float(np.mean(global_scores))

    sep_note = ' [Demucs]' if separate_vocals else ''

    # Timeline plot — 4 modele + ensemble
    fig_tl, ax = plt.subplots(figsize=(11, 3))
    for i, (f, color, label) in enumerate(zip(fs, _MODEL_COLORS, _MODEL_LABELS)):
        ax.plot(ts, f, color=color, linewidth=1.2, alpha=0.8,
                label=f'{label}: {global_scores[i]:.1%}')
    ax.fill_between(ts, fs_ens, alpha=0.15, color='#2c3e50')
    ax.plot(ts, fs_ens, color='#2c3e50', linewidth=2.2, linestyle='--',
            label=f'Ensemble: {score_ens:.1%}')
    ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.6)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Timp (s)')
    ax.set_ylabel('P(AI-generat)')
    verdict_str = 'AI-generat' if score_ens > 0.5 else 'Real'
    ax.set_title(f'Timeline Deepfake — {verdict_str} (Ensemble: {score_ens:.1%}){sep_note}')
    ax.legend(fontsize=7.5, ncol=2)
    fig_tl.tight_layout()

    # Spectrogram + overlay ensemble
    fig_sp, (ax_s, ax_p) = plt.subplots(2, 1, figsize=(12, 6),
                                          gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
    S = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE_DF, n_mels=128, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, sr=SAMPLE_RATE_DF, x_axis='time', y_axis='mel',
                                    fmax=8000, ax=ax_s, cmap='magma')
    fig_sp.colorbar(img, ax=ax_s, format='%+2.0f dB')
    for t, score in zip(ts, fs_ens):
        ax_s.axvspan(t - HOP_S / 2, t + HOP_S / 2, alpha=0.28,
                     color=plt.cm.RdYlGn_r(score), linewidth=0)
    ax_s.set_title('Mel Spectrogram + Overlay Ensemble' + sep_note)
    ax_p.plot(ts, fs_ens, color='#2c3e50', linewidth=1.5)
    ax_p.fill_between(ts, fs_ens, alpha=0.25, color='#2c3e50')
    ax_p.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, alpha=0.7)
    ax_p.set_ylim(0, 1)
    ax_p.set_ylabel('P(fake)')
    ax_p.set_xlabel('Timp (s)')
    fig_sp.tight_layout()

    return global_scores[0], global_scores[1], global_scores[2], global_scores[3], score_ens, fig_tl, fig_sp

print('detect_deepfake() definit (ensemble 4 modele + Demucs).')

In [ ]:
import sys, os, soundfile as sf, tempfile, numpy as np
from pathlib import Path

os.environ['weight_root'] = str(MODELS_DIR)
os.environ['index_root'] = str(MODELS_DIR)
os.environ['rmvpe_root'] = f'{RVC_DIR}/assets/rmvpe'

_rvc_instance = None
_rvc_current_voice = None

def voice_convert(
    audio_path,
    voice_id,
    transpose=0,
    index_rate=0.75,
    protect=0.33,
):
    global _rvc_instance, _rvc_current_voice
    cfg = VOICE_CONFIG[voice_id]
    model_file = cfg['model_file']
    model_path = str(MODELS_DIR / model_file)
    index_path = str(MODELS_DIR / cfg['index_file'])
    if not Path(model_path).exists():
        raise FileNotFoundError('Model negasit: ' + model_path)
    if _rvc_current_voice != voice_id:
        os.chdir(RVC_DIR)
        _orig = sys.argv[:]
        sys.argv = ['']
        from infer.modules.vc.modules import VC
        from configs.config import Config
        cfg_rvc = Config()
        cfg_rvc.is_half = False
        sys.argv = _orig
        _rvc_instance = VC(cfg_rvc)
        _rvc_instance.get_vc(model_file)
        _rvc_current_voice = voice_id
        print('Model incarcat: ' + cfg['name'])
    os.chdir(RVC_DIR)
    if Path(index_path).exists():
        use_index = index_path
    else:
        use_index = ''
    pitch = transpose + cfg.get('transpose', 0)
    result = _rvc_instance.vc_single(
        sid=0,
        input_audio_path=audio_path,
        f0_up_key=pitch,
        f0_file=None,
        f0_method='rmvpe',
        file_index=use_index,
        file_index2='',
        index_rate=index_rate,
        filter_radius=3,
        resample_sr=0,
        rms_mix_rate=0.25,
        protect=protect,
    )
    status_msg, audio_data = result
    if not isinstance(audio_data, tuple) or audio_data[0] is None:
        raise RuntimeError('vc_single error: ' + str(status_msg))
    tgt_sr, out = audio_data
    out = out.astype(np.float32) / 32768.0
    tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    sf.write(tmp.name, out, int(tgt_sr))
    return tmp.name

print('voice_convert OK')

In [ ]:
import gradio as gr, traceback

CUSTOM_CSS = """
.step-label {
    font-size: 0.85em; font-weight: 700; color: #6c63ff;
    text-transform: uppercase; letter-spacing: 0.08em; margin-bottom: 4px;
}
.voice-radio label {
    border: 2px solid #e2e8f0; border-radius: 10px;
    padding: 10px 16px; margin: 4px 0; transition: border-color 0.2s;
}
.voice-radio label:hover { border-color: #6c63ff; }
.verdict-box textarea {
    font-size: 1.3em !important; font-weight: 700;
    text-align: center; background: #f8f9fa; border-radius: 10px;
}
.alarm-box textarea {
    font-size: 1.0em !important; font-weight: 600;
    background: #fff5f5 !important; border: 2px solid #e74c3c !important;
    border-radius: 10px; color: #c0392b !important;
}
.convert-btn {
    height: 56px !important; font-size: 1.1em !important;
    border-radius: 12px !important;
}
footer { display: none !important; }
"""

VOICE_DESCRIPTIONS = {
    'voice_1': ('Calin Georgescu',  'Politician — ton grav, cadenta lenta'),
    'voice_2': ('Nicusor Dan',      'Primar Bucuresti — ton academic, clar'),
    'voice_3': ('Diana Sosoaca',    'Politician — ton ferm, energic'),
}

RADIO_CHOICES = [
    f'{name}  —  {desc}'
    for vid, (name, desc) in VOICE_DESCRIPTIONS.items()
]
RADIO_TO_VOICE_ID = {
    f'{name}  —  {desc}': vid
    for vid, (name, desc) in VOICE_DESCRIPTIONS.items()
}


def vc_callback_radio(audio_input, radio_choice, transpose, index_rate, protect):
    if audio_input is None:
        return None, 'Incarca un fisier audio mai intai.'
    if radio_choice is None:
        return None, 'Selecteaza o voce tinta.'
    voice_id = RADIO_TO_VOICE_ID.get(radio_choice)
    if voice_id is None:
        return None, 'Voce necunoscuta: ' + str(radio_choice)
    model_path = MODELS_DIR / VOICE_CONFIG[voice_id]['model_file']
    if not model_path.exists():
        return None, 'Modelul lipseste. Ruleaza 02_rvc_finetune.ipynb.'
    try:
        out = voice_convert(
            audio_input, voice_id,
            transpose=int(transpose),
            index_rate=float(index_rate),
            protect=float(protect),
        )
        name = VOICE_DESCRIPTIONS[voice_id][0]
        return out, 'Conversie completa: ' + name
    except Exception:
        return None, traceback.format_exc()


def _verdict_label(score):
    if score > 0.75:
        return f'AI-GENERAT ({score:.1%})'
    elif score > 0.5:
        return f'POSIBIL AI  ({score:.1%})'
    elif score > 0.25:
        return f'POSIBIL REAL ({score:.1%})'
    else:
        return f'REAL         ({score:.1%})'


def _alarm_check(s1, s2, s3, s4):
    """Returneaza semnal de alarma daca oricare model depaseste 70%."""
    model_names = ['M1 MelodyMachine', 'M2 Wav2Vec2-XLSR', 'M3 WavLM-base', 'M4 XLS-R-300M']
    scores = [s1, s2, s3, s4]
    triggered = [(name, score) for name, score in zip(model_names, scores) if score > 0.70]
    if not triggered:
        return ''
    lines = []
    for name, score in sorted(triggered, key=lambda x: -x[1]):
        if score >= 0.90:
            level = 'CRITIC'
        elif score >= 0.80:
            level = 'RIDICAT'
        else:
            level = 'MODERAT'
        lines.append(f'[{level}] {name}: {score:.1%}')
    header = 'SEMNAL DE ALARMA DETECTAT:\n'
    body = '\n'.join(lines)
    note = '\n\nAtentie: chiar daca scorul ensemble este sub 50%, un model specializat a detectat probabilitate mare de audio sintetic. Audioul necesita verificare suplimentara.'
    return header + body + note


def df_callback_new(audio_input, do_separate):
    if audio_input is None:
        return None, None, 'Incarca un fisier audio.', '', ''
    try:
        s1, s2, s3, s4, s_ens, fig_tl, fig_sp = detect_deepfake(
            audio_input, separate_vocals=bool(do_separate)
        )
        if s_ens > 0.75:
            verdict = f'PROBABIL AI-GENERAT\n{s_ens:.1%} probabilitate fake'
        elif s_ens > 0.5:
            verdict = f'POSIBIL AI-GENERAT\n{s_ens:.1%} probabilitate fake'
        elif s_ens > 0.25:
            verdict = f'POSIBIL REAL\n{s_ens:.1%} probabilitate fake'
        else:
            verdict = f'PROBABIL REAL\n{s_ens:.1%} probabilitate fake'

        sep_info = '  [Demucs]' if do_separate else ''
        details = (
            f'M1 MelodyMachine:   {_verdict_label(s1)}\n'
            f'M2 Wav2Vec2-XLSR:   {_verdict_label(s2)}\n'
            f'M3 WavLM-base:      {_verdict_label(s3)}\n'
            f'M4 XLS-R-300M:      {_verdict_label(s4)}\n'
            f'Ensemble (medie):   {_verdict_label(s_ens)}{sep_info}'
        )
        alarm = _alarm_check(s1, s2, s3, s4)
        return fig_tl, fig_sp, verdict, details, alarm
    except Exception:
        return None, None, traceback.format_exc(), '', ''


with gr.Blocks(
    theme=gr.themes.Soft(primary_hue='violet', neutral_hue='slate'),
    title='Neural Voice Identity Control',
    css=CUSTOM_CSS,
) as app:

    gr.HTML("""
    <div style="text-align:center; padding:28px 0 12px 0;">
      <h1 style="font-size:2em; font-weight:800; margin:0; color:#1e293b;">
        Neural Voice Identity Control
      </h1>
      <p style="color:#64748b; margin:6px 0 0 0; font-size:1em;">
        Deepfake Audio Analysis System &nbsp;&middot;&nbsp; Proiect academic 2026
      </p>
    </div>
    """)

    with gr.Tabs():

        with gr.Tab('Conversie Vocala'):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1, min_width=320):
                    gr.HTML('<p class="step-label">Pasul 1 — Incarca audio</p>')
                    vc_input = gr.Audio(
                        label='', sources=['upload', 'microphone'], type='filepath',
                    )
                    gr.HTML('<p class="step-label" style="margin-top:18px;">Pasul 2 — Alege vocea</p>')
                    vc_voice_radio = gr.Radio(
                        choices=RADIO_CHOICES,
                        value=RADIO_CHOICES[0] if RADIO_CHOICES else None,
                        label='', elem_classes='voice-radio',
                    )
                    with gr.Accordion('Setari avansate', open=False):
                        vc_transpose  = gr.Slider(label='Pitch shift (semitonuri)', minimum=-12, maximum=12, step=1, value=0)
                        vc_index_rate = gr.Slider(label='Index rate',               minimum=0.0, maximum=1.0, step=0.05, value=0.75)
                        vc_protect    = gr.Slider(label='Consonant protection',     minimum=0.0, maximum=0.5, step=0.01, value=0.33)
                    vc_btn = gr.Button('Converteste vocea', variant='primary', size='lg', elem_classes='convert-btn')
                with gr.Column(scale=1, min_width=320):
                    gr.HTML('<p class="step-label">Pasul 3 — Asculta si descarca</p>')
                    vc_output = gr.Audio(label='', type='filepath', interactive=False, show_download_button=True)
                    vc_status = gr.Textbox(label='Status', interactive=False, lines=6, placeholder='Asteapta conversia...')
            vc_btn.click(
                fn=vc_callback_radio,
                inputs=[vc_input, vc_voice_radio, vc_transpose, vc_index_rate, vc_protect],
                outputs=[vc_output, vc_status],
            )

        with gr.Tab('Detectie Deepfake'):
            with gr.Row(equal_height=False):
                with gr.Column(scale=1, min_width=300):
                    gr.HTML('<p class="step-label">Pasul 1 — Incarca audio</p>')
                    df_input = gr.Audio(label='', sources=['upload', 'microphone'], type='filepath')
                    df_separate = gr.Checkbox(
                        label='Separa vocea inainte de analiza (Demucs)',
                        value=False,
                        info='Recomandat pentru audio cu muzica de fundal. Adauga ~30-60s.',
                    )
                    df_btn = gr.Button('Analizeaza', variant='primary', size='lg', elem_classes='convert-btn')

                    gr.HTML('<p class="step-label" style="margin-top:18px;">Verdict Ensemble</p>')
                    df_verdict = gr.Textbox(
                        label='', interactive=False, lines=3,
                        elem_classes='verdict-box', placeholder='Asteapta analiza...',
                    )

                    gr.HTML('<p class="step-label" style="margin-top:12px;">Modele de comparatie</p>')
                    df_details = gr.Textbox(
                        label='', interactive=False, lines=5,
                        placeholder='M1 / M2 / M3 / M4 / Ensemble',
                    )

                    gr.HTML('<p class="step-label" style="margin-top:12px; color:#e74c3c;">Semnal de alarma (model individual > 70%)</p>')
                    df_alarm = gr.Textbox(
                        label='', interactive=False, lines=5,
                        elem_classes='alarm-box',
                        placeholder='Niciun model individual nu a depasit pragul de 70%.',
                    )

                with gr.Column(scale=2):
                    df_timeline    = gr.Plot(label='Analiza temporala (4 modele + ensemble)')
                    df_spectrogram = gr.Plot(label='Spectrograma Mel')

            df_btn.click(
                fn=df_callback_new,
                inputs=[df_input, df_separate],
                outputs=[df_timeline, df_spectrogram, df_verdict, df_details, df_alarm],
            )

        with gr.Tab('Despre'):
            gr.Markdown("""
## Despre sistem

Proiect disertatie: **Neural Voice Identity Control and Deepfake Audio Analysis System** (2026).

**Voice Conversion** — transforma timbrul vocal folosind RVC v2 (3 voci romanesti fine-tuned).

**Deepfake Detection** — ensemble de **4 modele** cu arhitecturi diferite:
| # | Model | Arhitectura | Antrenat pe |
|---|---|---|---|
| M1 | MelodyMachine/Deepfake-audio-detection-V2 | wav2vec2-base | General speech deepfakes |
| M2 | garystafford/wav2vec2-deepfake-voice-detector | wav2vec2-XLSR-53 | ElevenLabs, Amazon Polly, Speechify |
| M3 | DavidCombei/wavLM-base-Deepfake_V2 | WavLM-base (Microsoft) | ASVspoof + diverse |
| M4 | Gustking/wav2vec2-large-xlsr-deepfake-audio | XLS-R-300M | ASVspoof2019 |

**Semnal de alarma** — daca oricare model individual depaseste 70%, se afiseaza o alerta separata cu nivelul de severitate (MODERAT 70-79%, RIDICAT 80-89%, CRITIC 90%+), independent de scorul ensemble.

**Separare vocala Demucs** — extrage vocea din audio cu muzica de fundal inainte de detectie.

Sistemul e construit exclusiv in scop academic.
            """)

print('Gradio app definita (ensemble 4 modele + alarm system).')

In [ ]:
app.queue().launch(share=True, debug=False, quiet=False)